In [ ]:
import time
import pandas as pd
import numpy as np
from tqdm import tqdm
from datasets import load_metric
from transformers import MarianMTModel, MarianTokenizer
from comet import download_model, load_from_checkpoint
import torch
import time
from tqdm import tqdm


torch.backends.mps.is_available = lambda: False 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)
model.eval()

print(f"Loaded translation model: {model_name}")

df = pd.read_csv("OpenSubtitles+IWSLT_en-fr_clean.csv")
df = pd.read_csv("OpenSubtitles+IWSLT_en-fr_clean.csv")

opensubs_300 = df[df["dataset"]=="opensubs"].sample(n=300, random_state=42)
iwslt_300    = df[df["dataset"]=="iwslt2017_test"].sample(n=300, random_state=42)

df = pd.concat([opensubs_300, iwslt_300], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(df["dataset"].value_counts())


def translate_waitk_sentence(model, tokenizer, sentence, k, device, max_target_len=128):

    enc = tokenizer(sentence, return_tensors="pt", truncation=True).to(device)
    input_ids_full = enc["input_ids"][0]
    src_len = input_ids_full.size(0)

    if src_len == 0:
        return "", [], 0, 0

    j = min(k, src_len)

    start_id = model.config.decoder_start_token_id
    generated_ids = torch.tensor([[start_id]], device=device)

    g_list = []

    while True:
        encoder_input_ids = input_ids_full[:j].unsqueeze(0)
        attention_mask = torch.ones_like(encoder_input_ids, device=device)

        outputs = model(
            input_ids=encoder_input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=generated_ids
        )
        next_token_logits = outputs.logits[:, -1, :]
        next_token_id = next_token_logits.argmax(dim=-1, keepdim=True)

        generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)

        g_list.append(j)

        if next_token_id.item() == tokenizer.eos_token_id:
            break
        if len(g_list) >= max_target_len:
            break
        if j < src_len:
            j += 1

    final_ids = generated_ids[0, 1:]
    tgt_len = len(final_ids)

    translated = tokenizer.decode(final_ids, skip_special_tokens=True)
    return translated, g_list, src_len, tgt_len
def compute_average_lagging(g_list, src_len, tgt_len):

    if src_len == 0 or tgt_len == 0 or len(g_list) == 0:
        return 0.0

    r = tgt_len / src_len 
    tau = None
    for t, g in enumerate(g_list, start=1):
        if g == src_len:
            tau = t
            break
    if tau is None:
        tau = len(g_list)

    total = 0.0
    for t in range(1, tau + 1):
        g = g_list[t - 1]
        total += g - (t - 1) / r

    return total / tau

waitk_values = [1, 3, 5, 7]
results = {}

for k in waitk_values:
    preds = []
    al_list = []
    print(f"\nTranslating with wait-k={k} ...")

    for src in tqdm(df["src"], desc=f"wait-k={k}"):
        pred, g_list, src_len, tgt_len = translate_waitk_sentence(
            model, tokenizer, src, k, device=device
        )
        preds.append(pred)

        al = compute_average_lagging(g_list, src_len, tgt_len)
        al_list.append(al)

    df_k = pd.DataFrame({
        "src": df["src"],
        "tgt": df["tgt"],
        "pred": preds,
        "AL": al_list
    })

    output_path = f"OpenSubtitles_waitk{k}_translated_with_AL.csv"
    df_k.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Saved translated results with AL to {output_path}")

    results[k] = df_k

metric_bleu = load_metric("sacrebleu")
metric_chrf = load_metric("chrf")

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

final_scores = []

for k, df_k in results.items():
    print(f"\nEvaluating wait-k={k} ...")

    src_texts = df_k["src"].tolist()
    tgt_texts = df_k["tgt"].tolist()
    preds = df_k["pred"].tolist()

    bleu = metric_bleu.compute(predictions=preds, references=[[t] for t in tgt_texts])
    chrf = metric_chrf.compute(predictions=preds, references=[[t] for t in tgt_texts])

    data = [{"src": s, "mt": p, "ref": t} for s, p, t in zip(src_texts, preds, tgt_texts)]
    comet_score = comet_model.predict(data, batch_size=4, gpus=0, num_workers=0)
    comet_mean = np.mean(comet_score["system_score"])

    avg_al = df_k["AL"].mean()

    final_scores.append({
        "wait_k": k,
        "BLEU": bleu["score"],
        "chrF": chrf["score"],
        "COMET": comet_mean,
        "AverageLagging": avg_al,
    })

    print(
        f"wait-k={k}: BLEU={bleu['score']:.2f}, chrF={chrf['score']:.2f}, "
        f"COMET={comet_mean:.4f}, AL={avg_al:.2f}"
    )

results_df = pd.DataFrame(final_scores)
results_df.to_csv("OpenSubtitles_waitk_evaluation_summary_with_AL.csv", index=False)
print("\n===== Final Evaluation Summary (with AL) =====")
print(results_df)



/opt/anaconda3/envs/ctx_env/lib/python3.10/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loaded translation model: Helsinki-NLP/opus-mt-en-fr
dataset
opensubs          300
iwslt2017_test    300
Name: count, dtype: int64

Translating with wait-k=1 ...


wait-k=1: 100%|██████████| 600/600 [07:59<00:00,  1.25it/s]


Saved translated results with AL to OpenSubtitles_waitk1_translated_with_AL.csv

Translating with wait-k=3 ...


wait-k=3: 100%|██████████| 600/600 [07:21<00:00,  1.36it/s]


...

wait-k=3: BLEU=25.40, chrF=55.78, COMET=0.6736, AL=4.22

Evaluating wait-k=5 ...


Predicting DataLoader 0: 100%|██████████| 150/150 [01:31<00:00,  1.63it/s]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores


wait-k=5: BLEU=27.20, chrF=56.61, COMET=0.7205, AL=5.79

Evaluating wait-k=7 ...


Predicting DataLoader 0: 100%|██████████| 150/150 [01:26<00:00,  1.73it/s]

wait-k=7: BLEU=30.16, chrF=57.96, COMET=0.7429, AL=7.43

===== Final Evaluation Summary (with AL) =====
   wait_k       BLEU       chrF     COMET  AverageLagging
0       1  22.047818  54.178208  0.599492        2.877631
1       3  25.397853  55.781440  0.673628        4.220449
2       5  27.198312  56.606885  0.720509        5.787967
3       7  30.156427  57.964424  0.742890        7.426414
